## Cell 1 · Project configuration and output paths

In [ ]:
import os, sys
from pathlib import Path

# ══════════════ Student model ══════════════
# Default per the project brief. Cell 5 first looks for a student model in your project's config files and
# uses THAT if one is found (it never substitutes silently: whatever is used is printed).
STUDENT_MODEL_ID            = "Qwen/Qwen3-4B"
STUDENT_MODEL_REVISION      = None        # pin a commit SHA for strict reproducibility (None = current main)
USE_PROJECT_CONFIG_IF_FOUND = True
MODEL_VARIANT               = "base_student"   # stored in every output; use "distilled_student" in the later run

# ══════════════ Evaluation data ══════════════
EVAL_CSV_NAME     = "eval_prompts_60.csv"
EVAL_CSV_PATH     = None      # set an explicit path to skip the automatic search
EVAL_CSV_URL      = "https://drive.google.com/file/d/1VBq4DPk2Md6151JmjHNXQhEPu8x-lg8N/view?usp=sharing"  # used only if the file is not found locally
EXPECTED_TOTAL    = 60
EXPECTED_PER_TASK = 15
ALLOW_COUNT_MISMATCH = False  # if the CSV is not 60 rows / 15 per task, generation is blocked unless you set True
COLUMN_MAP        = {}        # manual schema override, e.g. {"id": "qid", "prompt": "question", "reference": "gold", "task": "category"}

# ══════════════ Model loading ══════════════
# "nf4"  : 4-bit NF4 (bitsandbytes). Default, because the later QLoRA student is a 4-bit base too, so the
#          before/after comparison then differs ONLY by the adapter. Also safest on a 16 GB T4.
# "none" : fp16/bf16 weights (~8 GB for a 4B model). Only if you have >= ~14 GB VRAM.
QUANTIZATION = "nf4"

# ══════════════ Shared decoding protocol (reuse UNCHANGED for the distilled model) ══════════════
# Greedy decoding is deterministic. When DO_SAMPLE=False, TEMPERATURE / TOP_P / TOP_K / MIN_P are ignored by
# generate() (they only shape the sampling distribution), so they are not applied or recorded as active.
DO_SAMPLE          = False
TEMPERATURE        = 0.7      # used only if DO_SAMPLE=True (Qwen's non-thinking recommendation)
TOP_P              = 0.8
TOP_K              = 20
MIN_P              = 0.0
REPETITION_PENALTY = 1.0
SEED               = 42
# Ge'ez script costs many tokens per word in Qwen's tokenizer, so limits are generous. Outputs that still hit the
# limit are KEPT but flagged "truncated" for the reviewer; there is no automatic retry, so every run is identical.
MAX_NEW_TOKENS_DEFAULT = 768
MAX_NEW_TOKENS_BY_TASK = {"closed_book_knowledge": 768, "instruction_following": 1024, "translation": 768, "summarization": 768}
MAX_INPUT_TOKENS   = 6144
SYSTEM_PROMPT      = None     # None = no system message (matches the student prompts used for SFT)
PROMPT_FORMAT_VERSION = "v1-sft-aligned"
BATCH_SIZE         = 1        # 1 is the most reliable on a T4. >1 may change greedy outputs very slightly (padding numerics).
RETRY_FAILED_ON_RESUME = True

# ══════════════ Output location ══════════════
RESULTS_SUBDIR = "baseline_results"      # the later distilled run should use e.g. "distilled_results"

# ══════════════ Paths (Colab -> Drive, Kaggle -> /kaggle/working, else local) ══════════════
IN_COLAB = False
try:
    import google.colab  # noqa: F401
    IN_COLAB = True
except ImportError:
    pass
IN_KAGGLE = (not IN_COLAB) and os.path.isdir("/kaggle/working")

if IN_COLAB:
    PROJECT_DIR = Path("/content/drive/MyDrive/amharic_distillation")
elif IN_KAGGLE:
    PROJECT_DIR = Path("/kaggle/working/amharic_distillation")
else:
    PROJECT_DIR = Path.cwd() / "amharic_distillation"

EVAL_INPUT_DIR   = PROJECT_DIR / "eval"
RESULTS_DIR      = PROJECT_DIR / "outputs" / RESULTS_SUBDIR
PRED_JSONL       = RESULTS_DIR / "predictions.jsonl"
PRED_CSV         = RESULTS_DIR / "predictions.csv"
SCORING_TEMPLATE = RESULTS_DIR / "scoring_template.csv"
RUN_CONFIG_PATH  = RESULTS_DIR / "run_config.json"
ERRORS_JSONL     = RESULTS_DIR / "generation_errors.jsonl"
SUMMARY_JSON     = RESULTS_DIR / "summary.json"
TASK_SUMMARY_CSV = RESULTS_DIR / "task_summary.csv"
CHART_PNG        = RESULTS_DIR / "task_scores.png"

print(f"Platform: {'Colab' if IN_COLAB else 'Kaggle' if IN_KAGGLE else 'local'}")
print(f"Project dir : {PROJECT_DIR}")
print(f"Results dir : {RESULTS_DIR}")

## Cell 2 · Google Drive mounting and where the CSV goes

In [ ]:
if IN_COLAB:
    from google.colab import drive
    drive.mount("/content/drive")
elif IN_KAGGLE:
    print("Kaggle: /kaggle/working is only kept if you 'Save Version'. Download predictions/scores before the session ends.")

for d in (PROJECT_DIR, EVAL_INPUT_DIR, RESULTS_DIR):
    d.mkdir(parents=True, exist_ok=True)

def find_local_eval_csv():
    """Look for the evaluation CSV without modifying anything. Returns Path or None."""
    if EVAL_CSV_PATH:
        p = Path(EVAL_CSV_PATH)
        if not p.is_file():
            raise FileNotFoundError(f"EVAL_CSV_PATH does not exist: {p}")
        return p
    hits = []
    for root, dirs, files in os.walk(PROJECT_DIR):
        dirs[:] = [d for d in dirs if d not in {"outputs", "teacher_outputs", "final_sft", ".git"}]
        if EVAL_CSV_NAME in files:
            hits.append(Path(root) / EVAL_CSV_NAME)
    for extra in (Path.cwd(), Path("/content")):
        if (extra / EVAL_CSV_NAME).is_file():
            hits.append(extra / EVAL_CSV_NAME)
    if os.path.isdir("/kaggle/input"):
        for root, _, files in os.walk("/kaggle/input"):
            if EVAL_CSV_NAME in files:
                hits.append(Path(root) / EVAL_CSV_NAME)
    uniq = {}
    for h in hits:
        uniq[str(h.resolve())] = h
    hits = list(uniq.values())
    if len(hits) > 1:
        raise RuntimeError("Several copies of the evaluation CSV were found; set EVAL_CSV_PATH to the right one:\n  "
                           + "\n  ".join(map(str, hits)))
    return hits[0] if hits else None

_found = find_local_eval_csv()
print("Evaluation CSV:", _found if _found else "NOT FOUND locally (Cell 6 will download it or ask for an upload)")
excl = [p for p in PROJECT_DIR.rglob("eval_exclusion_set.csv")] if PROJECT_DIR.exists() else []
if excl:
    print("Note: eval_exclusion_set.csv exists at", excl[0], "- it is NOT used by this notebook (it is not the benchmark).")

## Cell 3 · Dependencies and version checks

In [ ]:
!pip install -q -U "transformers>=4.57" accelerate bitsandbytes tqdm pandas matplotlib gdown huggingface_hub pyyaml

import importlib.metadata as md_
from packaging.version import Version
for pkg in ["transformers", "accelerate", "bitsandbytes", "huggingface_hub", "pandas", "matplotlib", "gdown"]:
    try:
        print(f"{pkg:16s}{md_.version(pkg)}")
    except md_.PackageNotFoundError:
        print(f"{pkg:16s}NOT INSTALLED")
assert Version(md_.version("transformers")) >= Version("4.51.0"), "Qwen3 needs transformers >= 4.51 (older versions raise KeyError: 'qwen3')."
# If pip upgraded a package that was already imported, restart the runtime (Runtime > Restart session) and re-run from Cell 1.

## Cell 4 · GPU and system inspection

In [ ]:
import subprocess, shutil, platform
import torch

print(subprocess.run(["nvidia-smi"], capture_output=True, text=True).stdout)
print("CUDA available:", torch.cuda.is_available())
if not torch.cuda.is_available():
    raise RuntimeError("No GPU detected. Colab: Runtime > Change runtime type > T4 GPU. Kaggle: Settings > Accelerator > GPU.")

props = torch.cuda.get_device_properties(0)
VRAM_GB       = props.total_memory / 1024**3
SUPPORTS_BF16 = props.major >= 8                       # Ampere+ only; a T4 (7.5) uses fp16
COMPUTE_DTYPE = torch.bfloat16 if SUPPORTS_BF16 else torch.float16
USE_4BIT      = (QUANTIZATION == "nf4")
GPU_NAME      = props.name

try:
    import psutil
    ram_gb = psutil.virtual_memory().total / 1024**3
except Exception:
    ram_gb = float("nan")

print(f"GPU            : {GPU_NAME}")
print(f"GPU memory     : {VRAM_GB:.1f} GB   (compute capability {props.major}.{props.minor})")
print(f"PyTorch        : {torch.__version__}")
print(f"System RAM     : {ram_gb:.1f} GB | Python {platform.python_version()}")
print(f"Inference dtype: {COMPUTE_DTYPE}")
print(f"Quantization   : {'4-bit NF4 (bitsandbytes), double-quant, compute dtype ' + str(COMPUTE_DTYPE) if USE_4BIT else 'none (full 16-bit weights)'}")

if QUANTIZATION not in ("nf4", "none"):
    raise ValueError("QUANTIZATION must be 'nf4' or 'none'.")
if not USE_4BIT and VRAM_GB < 14:
    raise RuntimeError(f"QUANTIZATION='none' needs ~14+ GB VRAM for a 4B model; this GPU has {VRAM_GB:.1f} GB. Use QUANTIZATION='nf4'.")
if VRAM_GB < 6:
    raise RuntimeError(f"Only {VRAM_GB:.1f} GB VRAM: too small even for a 4-bit 4B model plus KV cache.")
print("\nWhy this configuration: a 4B model in 4-bit uses roughly 3 GB, leaving ample room on a T4; and because the later "
      "QLoRA fine-tune also sits on a 4-bit base, evaluating the baseline in the same precision keeps the comparison fair.")

## Cell 5 · Imports, student-model resolution, configuration validation

In [ ]:
import re, json, time, gc, zlib, copy, hashlib, traceback
from datetime import datetime, timezone
from collections import Counter

import numpy as np
import pandas as pd
import transformers
from tqdm.auto import tqdm
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig, set_seed
from huggingface_hub import HfApi

# ── 1. Does the project already name a student model? (never substitute silently) ──
STUDENT_KEYS = {"student_model_id", "student_model", "student_model_name", "student_model_name_or_path", "student_id", "student"}
SKIP_DIRS = {"outputs", "teacher_outputs", "final_sft", "prompts", "eval", ".git", "__pycache__", ".ipynb_checkpoints"}

def _walk_for_student(obj, path=""):
    if isinstance(obj, dict):
        for k, v in obj.items():
            if str(k).lower() in STUDENT_KEYS:
                if isinstance(v, str):
                    yield f"{path}/{k}", v
                elif isinstance(v, dict):
                    for kk in ("model_id", "model", "name", "model_name_or_path", "id"):
                        if isinstance(v.get(kk), str):
                            yield f"{path}/{k}/{kk}", v[kk]
                            break
            yield from _walk_for_student(v, f"{path}/{k}")
    elif isinstance(obj, list):
        for i, v in enumerate(obj):
            yield from _walk_for_student(v, f"{path}[{i}]")

def find_student_in_project_config(max_depth=3):
    found = []
    if not PROJECT_DIR.exists():
        return found
    for root, dirs, files in os.walk(PROJECT_DIR):
        depth = len(Path(root).relative_to(PROJECT_DIR).parts)
        dirs[:] = [d for d in dirs if d not in SKIP_DIRS] if depth < max_depth else []
        for fn in files:
            p = Path(root) / fn
            if p.suffix.lower() not in (".json", ".yaml", ".yml") or p.stat().st_size > 1_000_000:
                continue
            try:
                text = p.read_text(encoding="utf-8")
                data = json.loads(text) if p.suffix.lower() == ".json" else __import__("yaml").safe_load(text)
            except Exception:
                continue
            for where, val in _walk_for_student(data):
                if "/" in val and " " not in val:
                    found.append((str(p.relative_to(PROJECT_DIR)), where, val))
    return found

_cfg_hits = find_student_in_project_config()
if _cfg_hits:
    print("Student model mentioned in project config files:")
    for f, w, v in _cfg_hits:
        print(f"   {f}  {w}  ->  {v}")
    distinct = sorted({v for _, _, v in _cfg_hits})
    if len(distinct) > 1:
        raise ValueError(f"Project configs name different student models {distinct}. Set STUDENT_MODEL_ID in Cell 1 explicitly.")
    if USE_PROJECT_CONFIG_IF_FOUND:
        if distinct[0] != STUDENT_MODEL_ID:
            print(f"   Using the project's student model {distinct[0]!r} instead of the default {STUDENT_MODEL_ID!r}.")
        STUDENT_MODEL_ID = distinct[0]
else:
    print(f"No student model found in project config files; using the default {STUDENT_MODEL_ID!r}.")

# ── 2. Is the repo ID valid? ──
MODEL_COMMIT_SHA = None
try:
    _info = HfApi().model_info(STUDENT_MODEL_ID, revision=STUDENT_MODEL_REVISION)
    MODEL_COMMIT_SHA = _info.sha
    print(f"Hugging Face repo verified: {STUDENT_MODEL_ID} @ {MODEL_COMMIT_SHA}")
except Exception as e:
    if type(e).__name__ in ("RepositoryNotFoundError", "RevisionNotFoundError", "GatedRepoError"):
        raise ValueError(f"Hugging Face repo check failed for {STUDENT_MODEL_ID!r}: {type(e).__name__}. "
                         "Check the ID (and revision) in Cell 1.") from e
    print(f"WARNING: could not reach the Hub to verify the repo ({type(e).__name__}); continuing. Commit SHA will be unknown.")

# ── 3. Validate settings ──
assert BATCH_SIZE >= 1 and MAX_NEW_TOKENS_DEFAULT > 0 and MAX_INPUT_TOKENS > 0
assert all(v > 0 for v in MAX_NEW_TOKENS_BY_TASK.values())
assert RESULTS_DIR.is_dir()
set_seed(SEED)
CURRENT_BATCH_SIZE = BATCH_SIZE

print("\n" + "=" * 78)
print(f" BASELINE MODEL FOR THIS RUN : {STUDENT_MODEL_ID}")
print(f" Variant                     : {MODEL_VARIANT}  (original weights; no fine-tuning, no adapters)")
print(f" Decoding                    : {'sampling' if DO_SAMPLE else 'greedy (deterministic)'}, seed {SEED}")
print("=" * 78)

## Cell 6 · Load and validate the 60-question CSV

In [ ]:
EVAL_PATH = find_local_eval_csv()

def fetch_eval_csv():
    dest = EVAL_INPUT_DIR / EVAL_CSV_NAME
    try:
        import gdown
        out = gdown.download(url=EVAL_CSV_URL, output=str(dest), quiet=False, fuzzy=True)
        if out is None:
            raise RuntimeError("gdown returned no file")
        head = dest.read_bytes()[:512].lstrip().lower()
        if head.startswith(b"<!doctype") or head.startswith(b"<html"):
            dest.unlink()
            raise RuntimeError("the link returned an HTML page, not the CSV (share it as 'Anyone with the link')")
        return dest
    except Exception as e:
        print(f"Download from EVAL_CSV_URL failed: {type(e).__name__}: {e}")
    if IN_COLAB:
        from google.colab import files
        print("Please select eval_prompts_60.csv in the upload dialog ...")
        up = files.upload()
        if up:
            name = EVAL_CSV_NAME if EVAL_CSV_NAME in up else next(iter(up))
            shutil.move(name, dest)
            return dest
    raise FileNotFoundError(
        f"Evaluation CSV not found. Do ONE of: (a) put {EVAL_CSV_NAME} in {EVAL_INPUT_DIR}; (b) set EVAL_CSV_PATH in Cell 1; "
        "(c) share the Drive file as 'Anyone with the link' and keep EVAL_CSV_URL. The notebook will not invent a benchmark.")

if EVAL_PATH is None:
    EVAL_PATH = fetch_eval_csv()
    try:
        os.chmod(EVAL_PATH, 0o444)
    except Exception:
        pass

CSV_SHA256 = hashlib.sha256(Path(EVAL_PATH).read_bytes()).hexdigest()
print(f"Evaluation file: {EVAL_PATH}\nSHA-256        : {CSV_SHA256}")

def read_eval_csv(path):
    last = None
    for enc in ("utf-8-sig", "utf-16"):
        try:
            return pd.read_csv(path, encoding=enc, dtype=str, keep_default_na=False), enc
        except UnicodeError as e:
            last = e
    raise ValueError(f"Could not decode the CSV as UTF-8 or UTF-16 ({last}). Re-save it as 'CSV UTF-8' so Amharic is preserved.")

EVAL_DF, _enc = read_eval_csv(EVAL_PATH)          # read-only: the CSV on disk is never written to
print(f"Encoding       : {_enc}\nRecords        : {len(EVAL_DF)}\nColumns        : {list(EVAL_DF.columns)}")

# ── Schema resolution ──
ROLE_CANDIDATES = {
    "id":          ["id", "eval_id", "example_id", "question_id", "qid", "uid", "item_id"],
    "task":        ["task", "task_type", "task_name", "category", "type"],
    "subtask":     ["subtask", "sub_task", "sub_type"],
    "prompt":      ["prompt", "question", "instruction", "query", "input"],
    "context":     ["context", "passage"],
    "source_text": ["source_text", "source", "text", "document"],
    "direction":   ["direction", "translation_direction", "lang_direction", "language_direction"],
    "reference":   ["reference_answer", "reference", "gold_answer", "gold", "expected_output", "expected_answer", "target", "answer"],
}
_lower = {c.strip().lower(): c for c in EVAL_DF.columns}
SCHEMA, _notes = {}, []
for role, cands in ROLE_CANDIDATES.items():
    if role in COLUMN_MAP:
        if COLUMN_MAP[role] not in EVAL_DF.columns:
            raise ValueError(f"COLUMN_MAP[{role!r}] = {COLUMN_MAP[role]!r} is not a column of the CSV: {list(EVAL_DF.columns)}")
        SCHEMA[role] = COLUMN_MAP[role]
        continue
    matches = [_lower[c] for c in cands if c in _lower]
    SCHEMA[role] = matches[0] if matches else None
    if len(matches) > 1:
        _notes.append(f"AMBIGUOUS {role}: columns {matches} all look like '{role}'. Using {matches[0]!r}. Override via COLUMN_MAP if wrong.")

print("\nSchema used (role -> CSV column):")
for role in ROLE_CANDIDATES:
    print(f"   {role:12s}-> {SCHEMA[role]!r}")
for n in _notes:
    print("   !!", n)
_unused = [c for c in EVAL_DF.columns if c not in set(v for v in SCHEMA.values() if v)]
print("Columns NOT used (never sent to the model):", _unused if _unused else "none")

if not SCHEMA["prompt"] and not SCHEMA["source_text"]:
    raise ValueError("No prompt/question column and no source_text column found. Set COLUMN_MAP['prompt'] in Cell 1.")
if not SCHEMA["task"]:
    raise ValueError("No task column found, so tasks cannot be scored separately. Set COLUMN_MAP['task'] in Cell 1.")

# ── Missing values / duplicates / references ──
print("\nMissing (empty) values per column:")
for c in EVAL_DF.columns:
    n_empty = int((EVAL_DF[c].str.strip() == "").sum())
    print(f"   {c:24s}{n_empty:4d}{'   <-- all empty' if n_empty == len(EVAL_DF) else ''}")

IDS_SYNTHETIC = SCHEMA["id"] is None
if IDS_SYNTHETIC:
    print("\n!! WARNING: no ID column. Synthetic IDs (row_001 ...) are used; they depend on row ORDER, so do not reorder the CSV.")
    _ids = [f"row_{i + 1:03d}" for i in range(len(EVAL_DF))]
else:
    _ids = [str(v).strip() for v in EVAL_DF[SCHEMA["id"]]]
_dup_ids = [k for k, c in Counter(_ids).items() if c > 1]
print(f"Duplicate IDs  : {_dup_ids if _dup_ids else 'none'}")

HAS_REFERENCES = bool(SCHEMA["reference"]) and int((EVAL_DF[SCHEMA["reference"]].str.strip() != "").sum()) > 0
print(f"Reference answers present: {HAS_REFERENCES}"
      + ("" if HAS_REFERENCES else "  (scoring will be rubric-only: reviewers have no gold answer to compare against)"))

# ── Canonical task groups ──
TASK_GROUPS = ["closed_book_knowledge", "instruction_following", "translation", "summarization"]

def canonical_task(raw):
    s = re.sub(r"[^a-z0-9]+", " ", str(raw).lower()).strip()
    if "summar" in s: return "summarization"
    if "translat" in s or s in {"mt", "en2am", "am2en", "en am", "am en"}: return "translation"
    if "instruct" in s: return "instruction_following"
    if "closed" in s or "knowledge" in s or "factual" in s or s in {"qa", "kb"}: return "closed_book_knowledge"
    return None

def _cell(row, role):
    col = SCHEMA.get(role)
    return str(row[col]) if col else ""

EVAL_RECORDS = []
for i, (_, row) in enumerate(EVAL_DF.iterrows()):
    raw_task = _cell(row, "task").strip()
    EVAL_RECORDS.append({
        "row_index": i, "id": _ids[i], "task_raw": raw_task,
        "task_group": canonical_task(raw_task) or raw_task,
        "task_mapped": canonical_task(raw_task) is not None,
        "subtask": _cell(row, "subtask").strip(), "direction": _cell(row, "direction").strip(),
        "prompt": _cell(row, "prompt"), "context": _cell(row, "context"), "source_text": _cell(row, "source_text"),
        "reference": _cell(row, "reference"),     # kept for scoring ONLY; never used to build model input
    })

print("\nSample records (first 3, text truncated to 90 chars):")
for r in EVAL_RECORDS[:3]:
    print("  ", {k: (v[:90] + "…" if isinstance(v, str) and len(v) > 90 else v) for k, v in r.items() if k != "reference"})

## Cell 7 · Task distribution and evaluation integrity

In [ ]:
BLOCKING_ISSUES, WARNINGS = [], []

dist = Counter(r["task_group"] for r in EVAL_RECORDS)
print(f"Total records: {len(EVAL_RECORDS)} (expected {EXPECTED_TOTAL})")
print(f"{'task group':28s}{'count':>6s}{'expected':>10s}")
for g in TASK_GROUPS:
    flag = "" if dist.get(g, 0) == EXPECTED_PER_TASK else "   <-- differs"
    print(f"{g:28s}{dist.get(g, 0):>6d}{EXPECTED_PER_TASK:>10d}{flag}")
for g in dist:
    if g not in TASK_GROUPS:
        print(f"{g:28s}{dist[g]:>6d}{'-':>10s}   <-- UNMAPPED task label")
        BLOCKING_ISSUES.append(f"task label {g!r} is not one of the four expected task groups (set COLUMN_MAP or fix the CSV)")

if len(EVAL_RECORDS) != EXPECTED_TOTAL or any(dist.get(g, 0) != EXPECTED_PER_TASK for g in TASK_GROUPS):
    msg = f"record counts differ from the expected {EXPECTED_TOTAL} ({EXPECTED_PER_TASK} per task)"
    (WARNINGS if ALLOW_COUNT_MISMATCH else BLOCKING_ISSUES).append(msg)

if _dup_ids:
    BLOCKING_ISSUES.append(f"duplicate IDs: {_dup_ids}")
_empty_ids = [r["row_index"] for r in EVAL_RECORDS if not r["id"]]
if _empty_ids:
    BLOCKING_ISSUES.append(f"empty IDs at rows {_empty_ids}")
_no_input = [r["id"] for r in EVAL_RECORDS if not (r["prompt"].strip() or r["source_text"].strip())]
if _no_input:
    BLOCKING_ISSUES.append(f"records with no prompt/source text: {_no_input}")
if IDS_SYNTHETIC:
    WARNINGS.append("IDs are synthetic (row order)")
if not HAS_REFERENCES:
    WARNINGS.append("no reference answers: rubric-only scoring")

print("\nFields that are SENT to the model : prompt, context / source_text (only where the task needs them)")
print("Fields NEVER sent to the model    : reference answer, ids, labels, every unmapped column")

# ── Leakage check against training material, if present in the project ──
def _norm(s): return re.sub(r"\s+", " ", str(s)).strip().lower()
_train_texts = []
for p in list((PROJECT_DIR / "prompts").glob("*.jsonl")) + [PROJECT_DIR / "final_sft" / "sft_dataset.jsonl"]:
    if not p.is_file():
        continue
    with open(p, encoding="utf-8-sig") as f:
        for line in f:
            try:
                o = json.loads(line)
            except Exception:
                continue
            for k in ("prompt", "source_text", "context"):
                if isinstance(o.get(k), str) and len(o[k]) >= 20:
                    _train_texts.append(_norm(o[k]))
_train_texts = list(set(_train_texts))
if _train_texts:
    leaks = []
    for r in EVAL_RECORDS:
        for t in (r["prompt"], r["source_text"], r["context"]):
            nt = _norm(t)
            if len(nt) >= 20 and any(nt == x or nt in x for x in _train_texts):
                leaks.append(r["id"])
                break
    print(f"\nTraining-overlap check vs {len(_train_texts)} training texts: " + (f"{len(leaks)} OVERLAPPING eval records {leaks}" if leaks else "no exact/substring overlap found"))
    if leaks:
        BLOCKING_ISSUES.append(f"eval items also appear in training material: {leaks}. Remove them from training before fine-tuning.")
else:
    print("\nTraining-overlap check skipped (no training prompt files found under the project dir).")

print("\n" + "=" * 70)
for w in WARNINGS:
    print("WARNING :", w)
for b in BLOCKING_ISSUES:
    print("BLOCKING:", b)
print("Integrity: " + ("OK to generate" if not BLOCKING_ISSUES else "FIX THE BLOCKING ISSUES ABOVE BEFORE RUNNING CELL 10"))

## Cell 8 · Load the ORIGINAL student model (no adapters)

In [ ]:
MODEL_LOADED = False
tokenizer = model = None
BNB_SETTINGS = None
try:
    tokenizer = AutoTokenizer.from_pretrained(STUDENT_MODEL_ID, revision=STUDENT_MODEL_REVISION)
    tokenizer.padding_side = "left"                    # needed for batched generation with decoder-only models
    load_kwargs = dict(revision=STUDENT_MODEL_REVISION, dtype=COMPUTE_DTYPE, device_map={"": 0})   # force GPU: fail loudly, never silently offload
    if USE_4BIT:
        BNB_SETTINGS = dict(load_in_4bit=True, bnb_4bit_quant_type="nf4", bnb_4bit_use_double_quant=True,
                            bnb_4bit_compute_dtype=str(COMPUTE_DTYPE))
        load_kwargs["quantization_config"] = BitsAndBytesConfig(
            load_in_4bit=True, bnb_4bit_quant_type="nf4", bnb_4bit_use_double_quant=True, bnb_4bit_compute_dtype=COMPUTE_DTYPE)
    model = AutoModelForCausalLM.from_pretrained(STUDENT_MODEL_ID, **load_kwargs)
    model.eval()
except torch.cuda.OutOfMemoryError as e:
    raise RuntimeError("CUDA out of memory while loading. Recovery: Runtime > Restart session (frees stale GPU memory), "
                       "re-run Cells 1-5, and keep QUANTIZATION='nf4'.") from e
except ImportError as e:
    raise RuntimeError(f"A dependency is missing or broken ({e}). Recovery: re-run Cell 3, then Runtime > Restart session.") from e
except Exception as e:
    raise RuntimeError(f"Model loading FAILED: {type(e).__name__}: {e}\n"
                       "Recovery options: (1) check internet / Hugging Face availability and the model ID in Cell 1; "
                       "(2) upgrade: pip install -U transformers bitsandbytes accelerate, then restart the session; "
                       "(3) if RAM/disk ran out, restart the session and retry.") from e

# ── Verify what was actually loaded (only then claim success) ──
assert model is not None and next(model.parameters()).is_cuda, "Model is not on the GPU."
assert not hasattr(model, "peft_config") and "peft" not in type(model).__module__.lower(), "A PEFT/LoRA adapter is attached: baseline must use the ORIGINAL weights."
if tokenizer.pad_token_id is None:
    tokenizer.pad_token = tokenizer.eos_token

GEN_BASE = copy.deepcopy(model.generation_config)
_eos = GEN_BASE.eos_token_id
EOS_IDS = set(_eos if isinstance(_eos, (list, tuple)) else [_eos])
_im_end = tokenizer.convert_tokens_to_ids("<|im_end|>")
if isinstance(_im_end, int) and _im_end != tokenizer.unk_token_id:
    EOS_IDS.add(_im_end)
EOS_IDS.discard(None)

def _valid_tok(t):
    i = tokenizer.convert_tokens_to_ids(t)
    return i if isinstance(i, int) and i != tokenizer.unk_token_id else None
THINK_START_ID, THINK_END_ID = _valid_tok("<think>"), _valid_tok("</think>")

_tpl = tokenizer.chat_template
_tpl_text = _tpl if isinstance(_tpl, str) else json.dumps(_tpl) if _tpl else ""
THINKING_SWITCH = "enable_thinking" in _tpl_text        # True for hybrid Qwen3 checkpoints
CHAT_TEMPLATE_SHA1 = hashlib.sha1(_tpl_text.encode("utf-8")).hexdigest()

n_params = sum(p.numel() for p in model.parameters())
MODEL_LOADED = True
print(f"Model loaded: {STUDENT_MODEL_ID}  (commit {MODEL_COMMIT_SHA})")
print(f"Class: {type(model).__name__} | parameters (as stored): {n_params/1e9:.2f} B | memory footprint: {model.get_memory_footprint()/1e9:.2f} GB")
print(f"GPU memory allocated: {torch.cuda.memory_allocated()/1e9:.2f} GB / {VRAM_GB:.1f} GB")
print(f"Quantization: {BNB_SETTINGS if USE_4BIT else 'none'} | adapter attached: False")
print(f"EOS ids: {sorted(EOS_IDS)} | pad id: {tokenizer.pad_token_id} | chat-template thinking switch: {THINKING_SWITCH}")
if THINKING_SWITCH:
    print("Thinking is disabled through enable_thinking=False in the chat template (the documented hard switch).")
else:
    print("This model's chat template has no thinking switch; nothing to disable.")

## Cell 9 · Prompt construction 

In [ ]:
LANG_NAMES = {"en": "English", "am": "Amharic"}
ETHIOPIC_RE = re.compile(r"[\u1200-\u139F\u2D80-\u2DDF\uAB00-\uAB2F]")

def ethiopic_ratio(text):
    letters = [c for c in text if c.isalpha()]
    return round(sum(1 for c in letters if ETHIOPIC_RE.match(c)) / len(letters), 3) if letters else None

def parse_direction(d):
    toks = re.sub(r"[^a-z]+", " ", str(d or "").lower()).split()
    en = [i for i, t in enumerate(toks) if t in ("english", "en")]
    am = [i for i, t in enumerate(toks) if t in ("amharic", "am")]
    if en and am:
        return ("English", "Amharic") if en[0] < am[0] else ("Amharic", "English")
    return None

def _attach(first, second):
    """first + blank line + second, unless `second` is already contained in `first`."""
    return f"{first}\n\n{second}" if second.strip() and second.strip() not in first else first

def build_user_prompt(rec):
    """Return (user_text, direction_source). Uses ONLY prompt / context / source_text."""
    g, p, ctx, src = rec["task_group"], rec["prompt"], rec["context"], rec["source_text"]
    if g == "closed_book_knowledge":
        return p, None                                               # closed-book: nothing but the question
    if g == "instruction_following":
        material = ctx or src
        return (f"{material}\n\n{p}" if material.strip() and material.strip() not in p else p), None
    if g == "translation":
        if p.strip():
            return _attach(p, src), None
        pair, how = parse_direction(rec["direction"]), "column"
        if pair is None:                                              # infer from the source script (project is EN<->AM only); flagged
            pair = ("Amharic", "English") if (ethiopic_ratio(src) or 0) > 0.5 else ("English", "Amharic")
            how = "inferred_from_source_script"
        return f"Translate the following {pair[0]} text into {pair[1]}.\n\n{src}", how
    if g == "summarization":
        material = src or ctx
        if p.strip():
            return _attach(p, material), None
        return f"Summarize the following text in Amharic.\n\n{material}", None
    raise ValueError(f"Cannot build a prompt for unmapped task group {g!r}")

TEMPLATE_KWARGS = {"enable_thinking": False} if THINKING_SWITCH else {}

def render_chat(user_text):
    msgs = ([{"role": "system", "content": SYSTEM_PROMPT}] if SYSTEM_PROMPT else []) + [{"role": "user", "content": user_text}]
    return tokenizer.apply_chat_template(msgs, tokenize=False, add_generation_prompt=True, **TEMPLATE_KWARGS)

def count_tokens(text):
    return len(tokenizer(text, add_special_tokens=False)["input_ids"])

# ── Precompute every prompt once ──
PROMPTS, _build_errors, _leaks, _inferred = {}, [], [], 0
for rec in EVAL_RECORDS:
    try:
        user, how = build_user_prompt(rec)
        rendered = render_chat(user)
        n_in = count_tokens(rendered)
    except Exception as e:
        _build_errors.append((rec["id"], f"{type(e).__name__}: {e}"))
        continue
    _inferred += how == "inferred_from_source_script"
    ref = rec["reference"].strip()
    if len(ref) >= 20 and ref in user:
        _leaks.append(rec["id"])
    PROMPTS[rec["id"]] = {
        "user": user, "rendered": rendered, "n_in": n_in, "direction_source": how,
        "max_new": MAX_NEW_TOKENS_BY_TASK.get(rec["task_group"], MAX_NEW_TOKENS_DEFAULT),
        "too_long": n_in > MAX_INPUT_TOKENS,
    }
if _build_errors:
    raise ValueError(f"Could not build prompts for: {_build_errors}")
if _leaks:
    raise ValueError(f"Reference answer text appears inside the model input for {_leaks}: refusing to continue (leakage).")
if _inferred:
    print(f"NOTE: translation direction was inferred from the source script for {_inferred} record(s) (no direction column / prompt).")
_long = [k for k, v in PROMPTS.items() if v["too_long"]]
print(f"Prompts built: {len(PROMPTS)} | token lengths: min {min(v['n_in'] for v in PROMPTS.values())}, "
      f"max {max(v['n_in'] for v in PROMPTS.values())} | over MAX_INPUT_TOKENS: {_long if _long else 'none'}")

print("\n— One rendered example per task (exactly what the model receives) —")
for g in TASK_GROUPS:
    r = next((x for x in EVAL_RECORDS if x["task_group"] == g), None)
    if r:
        print(f"\n[{g}] id={r['id']}\n{PROMPTS[r['id']]['rendered'][:700]}")
print("\nTail of the last rendered prompt (thinking disabled => empty think block expected for Qwen3):")
print(repr(PROMPTS[EVAL_RECORDS[-1]['id']]['rendered'][-90:]))

# ══════════════ Generation core ══════════════
def make_gen_config(max_new_tokens):
    g = copy.deepcopy(GEN_BASE)
    g.max_new_tokens = int(max_new_tokens)
    g.pad_token_id = tokenizer.pad_token_id
    g.eos_token_id = sorted(EOS_IDS)
    g.repetition_penalty = float(REPETITION_PENALTY)
    if DO_SAMPLE:
        g.do_sample, g.temperature, g.top_p, g.top_k = True, TEMPERATURE, TOP_P, TOP_K
        g.min_p = MIN_P if MIN_P else None
    else:
        g.do_sample = False
        g.temperature, g.top_p, g.top_k, g.min_p = 1.0, 1.0, 50, None     # neutral (ignored under greedy); overrides the model's shipped sampling defaults
    return g

@torch.inference_mode()
def run_generate(texts, max_new_tokens):
    if DO_SAMPLE:
        torch.manual_seed(SEED); torch.cuda.manual_seed_all(SEED)
    enc = tokenizer(texts, return_tensors="pt", padding=True, add_special_tokens=False).to(model.device)
    out = model.generate(**enc, generation_config=make_gen_config(max_new_tokens))
    rows = out[:, enc["input_ids"].shape[1]:].cpu().tolist()
    del enc, out
    results = []
    for ids in rows:
        cut, finished = len(ids), False
        for j, t in enumerate(ids):
            if t in EOS_IDS:
                cut, finished = j, True
                break
        ids = ids[:cut]
        think_state = "none"
        if THINK_END_ID is not None and THINK_END_ID in ids:             # discard everything up to the last </think>
            k = len(ids) - ids[::-1].index(THINK_END_ID)
            ids, think_state = ids[k:], "stripped"
        elif THINK_START_ID is not None and THINK_START_ID in ids:
            think_state = "unterminated"
        results.append({"raw": tokenizer.decode(ids, skip_special_tokens=True), "n_new": len(ids),
                        "finish": "stop" if finished else "length", "think_state": think_state})
    return results

def free_gpu():
    gc.collect(); torch.cuda.empty_cache()

def generate_robust(texts, max_new_tokens):
    """Never raises. OOM -> shrink batch size and bisect; other errors -> bisect to isolate the failing prompt."""
    global CURRENT_BATCH_SIZE
    err_text, oom = None, False
    try:
        return run_generate(texts, max_new_tokens)
    except Exception as e:                                               # noqa: BLE001
        err_text = f"{type(e).__name__}: {str(e)[:300]}"
        oom = isinstance(e, torch.cuda.OutOfMemoryError) or "out of memory" in str(e).lower()
    free_gpu()                                                           # outside the except block so the traceback is released first
    if oom and len(texts) > 1:
        CURRENT_BATCH_SIZE = max(1, min(CURRENT_BATCH_SIZE, len(texts) // 2))
        tqdm.write(f"[OOM] batch of {len(texts)} failed -> batch size reduced to {CURRENT_BATCH_SIZE}")
    if len(texts) == 1:
        return [{"exception": ("CUDA_OOM: " if oom else "generation_exception: ") + err_text}]
    mid = len(texts) // 2
    return generate_robust(texts[:mid], max_new_tokens) + generate_robust(texts[mid:], max_new_tokens)

def looks_repetitive(text):
    if len(text) < 200:
        return False
    b = text.encode("utf-8")
    if len(zlib.compress(b, 6)) / len(b) < 0.20:
        return True
    lines = [l.strip() for l in text.splitlines() if len(l.strip()) >= 20]
    return bool(lines) and max(Counter(lines).values()) >= 4

def evaluate_generation(res):
    """-> (answer_text or None, error or None, flags). Reasoning is never stored."""
    if "exception" in res:
        return None, res["exception"], []
    if res["think_state"] == "unterminated":
        return None, "unterminated_think_block", []
    text = re.sub(r"<think>.*?</think>", "", res["raw"], flags=re.S).strip()
    if "<think>" in text:
        return None, "unterminated_think_block", []
    if not text:
        return None, "empty_output", []
    flags = []
    if res["finish"] == "length": flags.append("truncated")
    if looks_repetitive(text):    flags.append("repetition")
    return text, None, flags

# ── Quick single-prompt smoke test (not saved) ──
_r0 = EVAL_RECORDS[0]
_t0 = time.time()
_res = generate_robust([PROMPTS[_r0["id"]]["rendered"]], 64)[0]
print(f"\nSmoke test on {_r0['id']} (first 64 new tokens, not saved): {time.time() - _t0:.1f}s")
print(_res.get("raw", _res.get("exception")))

## Cell 10 · Generate predictions

In [ ]:
def utc_now(): return datetime.now(timezone.utc).isoformat(timespec="seconds")
def sha1_text(s): return hashlib.sha1(str(s).encode("utf-8")).hexdigest()

def atomic_write_text(path, text):
    path = Path(path); tmp = path.with_name(path.name + ".tmp")
    with open(tmp, "w", encoding="utf-8", newline="\n") as f:
        f.write(text); f.flush(); os.fsync(f.fileno())
    os.replace(tmp, path)

def read_jsonl(path):
    recs, bad = [], []
    with open(path, "r", encoding="utf-8-sig") as f:
        for i, line in enumerate(f, 1):
            s = line.strip()
            if not s: continue
            try:
                o = json.loads(s)
                if not isinstance(o, dict): raise ValueError("not an object")
                recs.append(o)
            except Exception as e:                                     # noqa: BLE001
                bad.append((i, str(e), s))
    return recs, bad

def append_jsonl(path, recs):
    if not recs: return
    with open(path, "a", encoding="utf-8", newline="\n") as f:
        for r in recs:
            f.write(json.dumps(r, ensure_ascii=False) + "\n")          # Amharic stays readable
        f.flush(); os.fsync(f.fileno())

def _rank(r): return 1 if r.get("generation_status") == "success" else 0

def compact_jsonl(path):
    """Return {id: best_record}. Repairs the file atomically if it has a torn last line, duplicate IDs or no final newline.
    A success is never replaced by a failure."""
    path = Path(path)
    if not path.exists(): return {}
    recs, bad = read_jsonl(path)
    best, order, dups = {}, [], 0
    for r in recs:
        k = str(r.get("id"))
        if k in best:
            dups += 1
            if _rank(r) >= _rank(best[k]): best[k] = r
        else:
            best[k] = r; order.append(k)
    raw = path.read_bytes()
    if bad or dups or (raw and not raw.endswith(b"\n")):
        shutil.copy2(path, path.with_name(path.name + ".bak"))
        if bad:
            with open(path.with_name(path.name + ".corrupt"), "a", encoding="utf-8") as cf:
                for ln, err, line in bad:
                    cf.write(f"# line {ln}: {err}\n{line}\n")
        atomic_write_text(path, "".join(json.dumps(best[k], ensure_ascii=False) + "\n" for k in order))
        print(f"  [repair] {path.name}: removed {dups} duplicate line(s), quarantined {len(bad)} corrupt line(s).")
    return best

# ── Protocol fingerprints: what MUST stay identical between baseline and distilled runs ──
_decoding = {"do_sample": DO_SAMPLE, "repetition_penalty": REPETITION_PENALTY, "seed": SEED}
if DO_SAMPLE:
    _decoding.update(temperature=TEMPERATURE, top_p=TOP_P, top_k=TOP_K, min_p=MIN_P)
EVAL_PROTOCOL = {
    "csv_sha256": CSV_SHA256, "schema": SCHEMA, "expected_total": EXPECTED_TOTAL, "expected_per_task": EXPECTED_PER_TASK,
    "prompt_format_version": PROMPT_FORMAT_VERSION, "system_prompt": SYSTEM_PROMPT,
    "thinking_disabled_via_chat_template": THINKING_SWITCH, "decoding": _decoding,
    "max_new_tokens_by_task": {g: MAX_NEW_TOKENS_BY_TASK.get(g, MAX_NEW_TOKENS_DEFAULT) for g in TASK_GROUPS},
    "max_input_tokens": MAX_INPUT_TOKENS,
}
EVAL_PROTOCOL_SHA1 = sha1_text(json.dumps(EVAL_PROTOCOL, sort_keys=True, ensure_ascii=False))
MODEL_PROTOCOL = {"model_id": STUDENT_MODEL_ID, "requested_revision": STUDENT_MODEL_REVISION,
                  "quantization": QUANTIZATION, "compute_dtype": str(COMPUTE_DTYPE)}
EVAL_IDS_SHA1 = sha1_text("\n".join(r["id"] for r in EVAL_RECORDS))

# ── Run identity / resume safety ──
if BLOCKING_ISSUES:
    raise RuntimeError("Blocking integrity issues from Cell 7:\n  - " + "\n  - ".join(BLOCKING_ISSUES))

if RUN_CONFIG_PATH.exists():
    _old = json.loads(RUN_CONFIG_PATH.read_text(encoding="utf-8"))
    _bad = []
    if _old.get("eval_protocol_sha1") != EVAL_PROTOCOL_SHA1:
        _bad.append("evaluation protocol (CSV / prompt format / decoding / max tokens)")
    if _old.get("model_protocol") != MODEL_PROTOCOL:
        _bad.append("model / quantization")
    if _bad:
        raise RuntimeError("Existing results in " + str(RESULTS_DIR) + " were produced with a different " + " and ".join(_bad) +
                           ".\nMixing settings would invalidate the baseline. To intentionally start a NEW run, set a new RESULTS_SUBDIR "
                           "in Cell 1; to continue THIS run, restore the original settings.")
    RUN_ID, STARTED_AT = _old["run_id"], _old["started_at"]
    if _old.get("resolved_commit_sha") and MODEL_COMMIT_SHA and _old["resolved_commit_sha"] != MODEL_COMMIT_SHA:
        print("WARNING: the model repo has a newer commit than the one this run started with. Pin STUDENT_MODEL_REVISION if that matters.")
    print(f"Resuming existing run: {RUN_ID}")
else:
    RUN_ID = f"{MODEL_VARIANT}__{STUDENT_MODEL_ID.replace('/', '_')}__{datetime.now(timezone.utc).strftime('%Y%m%dT%H%M%SZ')}"
    STARTED_AT = utc_now()
    print(f"Starting new run: {RUN_ID}")

def write_run_config(status):
    cfg = {
        "run_id": RUN_ID, "model_variant": MODEL_VARIANT, "status": status, "started_at": STARTED_AT, "updated_at": utc_now(),
        "model_id": STUDENT_MODEL_ID, "requested_revision": STUDENT_MODEL_REVISION, "resolved_commit_sha": MODEL_COMMIT_SHA,
        "adapter_attached": False, "fine_tuned": False, "quantization": QUANTIZATION, "bitsandbytes_settings": BNB_SETTINGS,
        "compute_dtype": str(COMPUTE_DTYPE), "chat_template_sha1": CHAT_TEMPLATE_SHA1,
        "model_protocol": MODEL_PROTOCOL, "eval_protocol": EVAL_PROTOCOL, "eval_protocol_sha1": EVAL_PROTOCOL_SHA1,
        "eval_ids_sha1": EVAL_IDS_SHA1, "eval_csv_path": str(EVAL_PATH), "eval_csv_sha256": CSV_SHA256, "n_eval_records": len(EVAL_RECORDS),
        "ids_synthetic": IDS_SYNTHETIC, "has_reference_answers": HAS_REFERENCES,
        "batch_size_configured": BATCH_SIZE, "batch_size_current": CURRENT_BATCH_SIZE,
        "effective_generation_note": "temperature/top_p/top_k/min_p are ignored when do_sample=false",
        "eos_token_ids": sorted(EOS_IDS),
        "environment": {"python": platform.python_version(), "torch": torch.__version__, "transformers": transformers.__version__,
                        "gpu": GPU_NAME, "gpu_memory_gb": round(VRAM_GB, 1)},
    }
    if status == "completed":
        cfg["completed_at"] = utc_now()
    atomic_write_text(RUN_CONFIG_PATH, json.dumps(cfg, indent=2, ensure_ascii=False))

def make_prediction_record(rec, info, text, error, flags, res, secs, bs_used):
    return {
        "id": rec["id"], "task": rec["task_raw"], "task_group": rec["task_group"],
        "subtask": rec["subtask"] or None, "direction": rec["direction"] or None,
        "model_id": STUDENT_MODEL_ID, "model_variant": MODEL_VARIANT, "run_id": RUN_ID,
        "input_prompt": info["user"],
        "prediction": text,
        "generation_status": "success" if error is None else "failed",
        "generation_time_seconds": round(secs, 3), "error": error,
        "finish_reason": res.get("finish"), "input_tokens": info["n_in"], "new_tokens": res.get("n_new"),
        "max_new_tokens": info["max_new"], "flags": flags,
        "ethiopic_ratio": ethiopic_ratio(text) if text else None,
        "decoding": "sampling" if DO_SAMPLE else "greedy", "batch_size_used": bs_used, "generated_at": utc_now(),
    }

def export_predictions_csv():
    best = compact_jsonl(PRED_JSONL)
    rows = []
    for rec in EVAL_RECORDS:                                           # original CSV order
        r = best.get(rec["id"])
        if r:
            r = dict(r); r["flags"] = ";".join(r.get("flags") or [])
            rows.append(r)
    if rows:
        pd.DataFrame(rows).to_csv(PRED_CSV, index=False, encoding="utf-8-sig")   # BOM so Excel shows Amharic correctly
    return best

# ── Main loop ──
write_run_config("in_progress")
done = compact_jsonl(PRED_JSONL)
pending = [r for r in EVAL_RECORDS if (r["id"] not in done) or (RETRY_FAILED_ON_RESUME and done[r["id"]]["generation_status"] != "success")]
n_prev_ok = sum(1 for r in EVAL_RECORDS if done.get(r["id"], {}).get("generation_status") == "success")
print(f"Total {len(EVAL_RECORDS)} | already successful {n_prev_ok} | to generate now {len(pending)}")

ok_n = fail_n = 0
t_start = time.time()
pbar = tqdm(total=len(pending), desc="baseline generation", unit="prompt")
try:
    for g in TASK_GROUPS + [x for x in {r["task_group"] for r in pending} if x not in TASK_GROUPS]:   # one task at a time: each task has its own max_new_tokens
        todo = [r for r in pending if r["task_group"] == g]
        i = 0
        while i < len(todo):
            bs = max(1, min(CURRENT_BATCH_SIZE, len(todo) - i))
            batch = todo[i:i + bs]; i += len(batch)
            out_recs, runnable = [], []
            for rec in batch:
                info = PROMPTS[rec["id"]]
                if info["too_long"]:
                    out_recs.append(make_prediction_record(rec, info, None, f"input_too_long: {info['n_in']} > {MAX_INPUT_TOKENS} tokens", [], {}, 0.0, bs))
                else:
                    runnable.append(rec)
            if runnable:
                t0 = time.time()
                results = generate_robust([PROMPTS[r["id"]]["rendered"] for r in runnable], PROMPTS[runnable[0]["id"]]["max_new"])
                per = (time.time() - t0) / len(runnable)
                for rec, res in zip(runnable, results):
                    text, err, flags = evaluate_generation(res)
                    out_recs.append(make_prediction_record(rec, PROMPTS[rec["id"]], text, err, flags, res, per, len(runnable)))
            append_jsonl(PRED_JSONL, out_recs)                         # checkpoint immediately
            fails = [r for r in out_recs if r["generation_status"] != "success"]
            append_jsonl(ERRORS_JSONL, [{"timestamp": utc_now(), "run_id": RUN_ID, "id": r["id"], "error": r["error"]} for r in fails])
            ok_n += len(out_recs) - len(fails); fail_n += len(fails)
            pbar.update(len(out_recs))
            pbar.set_postfix(completed=ok_n, failed=fail_n, remaining=len(pending) - ok_n - fail_n, bs=CURRENT_BATCH_SIZE)
            if (ok_n + fail_n) % 10 == 0:
                free_gpu()
except KeyboardInterrupt:
    print("\nInterrupted. Everything generated so far is saved; re-run this cell to resume.")
finally:
    pbar.close()
    free_gpu()

best = export_predictions_csv()
_complete = all(r["id"] in best and best[r["id"]]["generation_status"] == "success" for r in EVAL_RECORDS)
write_run_config("completed" if _complete else "incomplete")
print(f"\nThis session: +{ok_n} success, +{fail_n} failed in {(time.time() - t_start) / 60:.1f} min")
print("Run status:", "COMPLETED: all prompts have a successful prediction" if _complete else "INCOMPLETE: re-run this cell to resume / retry failures")

## Cell 11 · Validate output completeness and uniqueness

In [ ]:
def validate_predictions():
    problems = []
    if not PRED_JSONL.exists():
        print("No predictions file yet."); return ["no predictions file"]
    recs, bad = read_jsonl(PRED_JSONL)
    ids_out = [str(r.get("id")) for r in recs]
    ids_in = [r["id"] for r in EVAL_RECORDS]
    dup = [k for k, c in Counter(ids_out).items() if c > 1]
    missing = [k for k in ids_in if k not in set(ids_out)]
    extra = [k for k in set(ids_out) if k not in set(ids_in)]
    ok = [r for r in recs if r.get("generation_status") == "success"]
    failed = [r for r in recs if r.get("generation_status") != "success"]
    empty_ok = [r["id"] for r in ok if not (isinstance(r.get("prediction"), str) and r["prediction"].strip())]
    wrong_run = [r["id"] for r in recs if r.get("model_id") != STUDENT_MODEL_ID]
    has_think = [r["id"] for r in ok if "<think>" in r["prediction"] or "</think>" in r["prediction"]]
    trunc = [r["id"] for r in ok if "truncated" in (r.get("flags") or [])]
    rep = [r["id"] for r in ok if "repetition" in (r.get("flags") or [])]

    rows = [
        ("input records", len(ids_in), False), ("output records", len(recs), False),
        ("successful", len(ok), False), ("failed", len(failed), len(failed) > 0),
        ("malformed JSON lines", len(bad), len(bad) > 0), ("duplicate output IDs", len(dup), len(dup) > 0),
        ("input IDs with no prediction", len(missing), len(missing) > 0), ("output IDs not in input", len(extra), len(extra) > 0),
        ("successful but empty prediction", len(empty_ok), len(empty_ok) > 0),
        ("records from a different model_id", len(wrong_run), len(wrong_run) > 0),
        ("predictions containing think tags", len(has_think), len(has_think) > 0),
        ("[info] truncated at max_new_tokens", len(trunc), False), ("[info] repetition flagged", len(rep), False),
    ]
    for name, n, is_problem in rows:
        print(f"  {'!!' if is_problem else 'OK'}  {name:40s}{n}")
        if is_problem: problems.append(name)
    for label, lst in (("failed", [r["id"] for r in failed]), ("missing", missing), ("duplicate", dup)):
        if lst: print(f"      {label} IDs: {lst[:15]}{' ...' if len(lst) > 15 else ''}")
    print("\nVALIDATION:", "PASS (complete and unique)" if not problems else "NOT COMPLETE: " + "; ".join(problems))
    return problems

PRED_PROBLEMS = validate_predictions()

## Cell 12 · Create the human scoring template


In [ ]:
RUBRIC_SCALE = {0: "Incorrect, irrelevant, missing, or unusable answer", 1: "Major errors; only a small part of the task is completed correctly",
                2: "Partially correct; meaningful omissions or errors remain", 3: "Mostly correct; minor errors or omissions",
                4: "Correct, relevant, and sufficiently complete answer"}
RUBRIC_CRITERIA = {
    "closed_book_knowledge": "Factual correctness; relevance to the question; completeness appropriate to the question; no unsupported claims.",
    "instruction_following": "Does the requested task; obeys format and length limits; uses the provided context correctly; adds nothing unsupported.",
    "translation": "Meaning preserved; fluent and natural; grammatical; key terminology correct; names, numbers, dates and key details preserved. Judge in the stated direction (EN->AM or AM->EN).",
    "summarization": "Covers the main ideas; faithful to the supplied text; keeps essential facts and relationships; concise without important omissions.",
}
TEMPLATE_COLUMNS = ["id", "task", "subtask", "prompt", "prediction", "reference_answer", "rubric_criteria",
                    "auto_flags", "score_0_to_4", "reviewer_comments", "prediction_sha1"]

def build_template_frame():
    best = compact_jsonl(PRED_JSONL)
    rows = []
    for rec in EVAL_RECORDS:
        p = best.get(rec["id"])
        pred = (p or {}).get("prediction") or ""
        flags = ";".join((p or {}).get("flags") or [])
        if not p:
            flags = "NO_PREDICTION_YET"
        elif p["generation_status"] != "success":
            flags = f"GENERATION_FAILED: {p.get('error')}"
        rows.append({
            "id": rec["id"], "task": rec["task_raw"], "subtask": rec["subtask"],
            "prompt": PROMPTS[rec["id"]]["user"], "prediction": pred, "reference_answer": rec["reference"],
            "rubric_criteria": RUBRIC_CRITERIA.get(rec["task_group"], ""), "auto_flags": flags,
            "score_0_to_4": "", "reviewer_comments": "", "prediction_sha1": sha1_text(pred),
        })
    return pd.DataFrame(rows, columns=TEMPLATE_COLUMNS)

def _has_scores(path):
    try:
        df = pd.read_csv(path, dtype=str, keep_default_na=False, encoding="utf-8-sig")
        return bool((df.get("score_0_to_4", pd.Series(dtype=str)).str.strip() != "").any())
    except Exception:
        return True            # unreadable file: be safe and do not overwrite it

_tpl_df = build_template_frame()
if SCORING_TEMPLATE.exists() and _has_scores(SCORING_TEMPLATE):
    alt = SCORING_TEMPLATE.with_name("scoring_template.regenerated.csv")
    _tpl_df.to_csv(alt, index=False, encoding="utf-8-sig")
    print(f"scoring_template.csv already contains scores: NOT overwritten.\nA fresh unscored copy was written to {alt.name} for reference.")
else:
    _tpl_df.to_csv(SCORING_TEMPLATE, index=False, encoding="utf-8-sig")
    print(f"Scoring template written: {SCORING_TEMPLATE}   ({len(_tpl_df)} rows; scores and comments left blank)")

print("\nRUBRIC (score each answer 0-4):")
for k, v in RUBRIC_SCALE.items():
    print(f"  {k} = {v}")
print("\nTask-specific criteria:")
for k, v in RUBRIC_CRITERIA.items():
    print(f"  {k}: {v}")
if not HAS_REFERENCES:
    print("\nLIMITATION: the CSV has no reference answers, so reviewers judge against the rubric and their own knowledge only.")
print(f"\nNEXT: score {SCORING_TEMPLATE.name} by hand (Google Sheets > File > Download > CSV), save it back under the same name, then run Cell 13.")

## Cell 13 · Load reviewed scores and calculate metrics

In [ ]:
REVIEWED_SCORES_PATH = SCORING_TEMPLATE       # or point to your reviewed copy
ALLOW_STALE_SCORES   = False                  # True = accept scores even if predictions were regenerated after scoring (not recommended)

def parse_score(v):
    """-> (int_or_None, error_or_None). Blank = not scored. Accepts 3 or 3.0; rejects 3.5, 5, -1, text."""
    s = "" if v is None else str(v).strip()
    if s == "" or s.lower() in ("nan", "none"):
        return None, None
    try:
        f = float(s)
    except ValueError:
        return None, f"not a number: {s!r}"
    if not f.is_integer():
        return None, f"not an integer: {s!r}"
    i = int(f)
    if not 0 <= i <= 4:
        return None, f"out of range 0-4: {s!r}"
    return i, None

def _block(scores, n_total):
    n = len(scores); s = sum(scores)
    return {"n_total": n_total, "n_scored": n, "n_missing": n_total - n, "sum_score": s,
            "max_possible_scored": 4 * n, "max_possible_total": 4 * n_total,
            "mean_0_4": round(s / n, 4) if n else None, "percent": round(100 * s / (4 * n), 2) if n else None,
            "distribution": {str(k): sum(1 for x in scores if x == k) for k in range(5)}}

def compute_metrics(rows):
    """rows: list of {'id', 'task_group', 'score' (int or None)}. Unscored items are NEVER counted as zero."""
    per_task, all_scores = {}, []
    groups = TASK_GROUPS + sorted({r["task_group"] for r in rows} - set(TASK_GROUPS))
    for g in groups:
        rs = [r for r in rows if r["task_group"] == g]
        if not rs: continue
        sc = [r["score"] for r in rs if r["score"] is not None]
        per_task[g] = _block(sc, len(rs)); all_scores += sc
    overall = _block(all_scores, len(rows))
    status = "not_started" if overall["n_scored"] == 0 else "complete" if overall["n_missing"] == 0 else "partial"
    return {"scoring_status": status, "overall": overall, "per_task": per_task}

METRICS = None
if not REVIEWED_SCORES_PATH.exists():
    print(f"No scoring file at {REVIEWED_SCORES_PATH}. Run Cell 12 first.")
else:
    sdf = pd.read_csv(REVIEWED_SCORES_PATH, dtype=str, keep_default_na=False, encoding="utf-8-sig")
    for c in ("id", "score_0_to_4"):
        if c not in sdf.columns:
            raise ValueError(f"Scoring file is missing the required column {c!r}. Columns found: {list(sdf.columns)}")
    ids = [str(x).strip() for x in sdf["id"]]
    dups = [k for k, c in Counter(ids).items() if c > 1]
    unknown = [k for k in ids if k not in {r["id"] for r in EVAL_RECORDS}]
    absent = [r["id"] for r in EVAL_RECORDS if r["id"] not in set(ids)]
    if dups or unknown or absent:
        raise ValueError(f"Scoring file IDs do not match the evaluation IDs. duplicates={dups} unknown={unknown} absent={absent}")

    parsed, invalid = {}, []
    for i, (k, v) in enumerate(zip(ids, sdf["score_0_to_4"])):
        sc, err = parse_score(v)
        if err: invalid.append((k, i + 2, err))          # +2 = spreadsheet row number (header is row 1)
        parsed[k] = sc
    if invalid:
        print("INVALID SCORES (fix them in the sheet, save, and re-run this cell):")
        for k, row, err in invalid:
            print(f"   id={k} (sheet row {row}): {err}")
        raise ValueError(f"{len(invalid)} invalid score(s): scores must be integers from 0 to 4 (or left blank).")

    # stale-score guard: were the scored predictions the ones currently on disk?
    best = compact_jsonl(PRED_JSONL)
    stale = []
    if "prediction_sha1" in sdf.columns:
        for k, h in zip(ids, sdf["prediction_sha1"]):
            cur = (best.get(k) or {}).get("prediction") or ""
            if h.strip() and h.strip() != sha1_text(cur) and parsed[k] is not None:
                stale.append(k)
    if stale:
        msg = f"{len(stale)} scored item(s) have predictions that changed since the template was created: {stale[:10]}"
        if not ALLOW_STALE_SCORES:
            raise ValueError(msg + ". Scores may refer to different outputs. Re-score, or set ALLOW_STALE_SCORES=True.")
        print("WARNING:", msg)

    rows = [{"id": r["id"], "task_group": r["task_group"], "score": parsed[r["id"]]} for r in EVAL_RECORDS]
    METRICS = compute_metrics(rows)
    o = METRICS["overall"]
    print(f"Scoring status: {METRICS['scoring_status'].upper()}   (scored {o['n_scored']} / {o['n_total']}, missing {o['n_missing']})")
    if METRICS["scoring_status"] == "partial":
        print("!! PARTIAL RESULT: averages use only the scored questions; unscored questions are NOT counted as zero.")
    print(f"\n{'task':26s}{'scored':>7s}{'missing':>8s}{'sum':>6s}{'max(scored)':>12s}{'mean/4':>8s}{'percent':>9s}   distribution 0..4")
    for g, b in METRICS["per_task"].items():
        d = b["distribution"]
        print(f"{g:26s}{b['n_scored']:>7d}{b['n_missing']:>8d}{b['sum_score']:>6d}{b['max_possible_scored']:>12d}"
              f"{(b['mean_0_4'] if b['mean_0_4'] is not None else float('nan')):>8.2f}{(b['percent'] if b['percent'] is not None else float('nan')):>8.1f}%   "
              + " ".join(f"{k}:{d[k]}" for k in "01234"))
    print(f"{'OVERALL':26s}{o['n_scored']:>7d}{o['n_missing']:>8d}{o['sum_score']:>6d}{o['max_possible_scored']:>12d}"
          f"{(o['mean_0_4'] if o['mean_0_4'] is not None else float('nan')):>8.2f}{(o['percent'] if o['percent'] is not None else float('nan')):>8.1f}%   "
          + " ".join(f"{k}:{o['distribution'][k]}" for k in "01234"))
    print(f"\nMax possible if everything were scored: {o['max_possible_total']} points ({EXPECTED_PER_TASK * 4} per task of {EXPECTED_PER_TASK} questions).")

## Cell 14 · Save summary tables and the task-score chart

In [ ]:
import matplotlib
import matplotlib.pyplot as plt

best = compact_jsonl(PRED_JSONL)
ok_recs = [r for r in best.values() if r.get("generation_status") == "success"]
GEN_STATS = {
    "n_eval_records": len(EVAL_RECORDS), "n_predictions": len(best), "n_success": len(ok_recs),
    "n_failed": sum(1 for r in best.values() if r.get("generation_status") != "success"),
    "n_missing": sum(1 for r in EVAL_RECORDS if r["id"] not in best),
    "total_generation_seconds": round(sum(r.get("generation_time_seconds") or 0 for r in ok_recs), 1),
    "mean_seconds_per_prediction": round(float(np.mean([r["generation_time_seconds"] for r in ok_recs])), 2) if ok_recs else None,
    "n_truncated": sum("truncated" in (r.get("flags") or []) for r in ok_recs),
    "n_repetition_flagged": sum("repetition" in (r.get("flags") or []) for r in ok_recs),
    "per_task_success": {g: sum(1 for r in ok_recs if r["task_group"] == g) for g in TASK_GROUPS},
}

SUMMARY = {
    "run_id": RUN_ID, "model_id": STUDENT_MODEL_ID, "model_variant": MODEL_VARIANT, "resolved_commit_sha": MODEL_COMMIT_SHA,
    "fine_tuned": False, "adapter_attached": False, "quantization": QUANTIZATION,
    "eval_csv_sha256": CSV_SHA256, "eval_ids_sha1": EVAL_IDS_SHA1, "eval_protocol_sha1": EVAL_PROTOCOL_SHA1,
    "generated_summary_at": utc_now(), "generation": GEN_STATS,
    "scoring_status": (METRICS or {}).get("scoring_status", "not_started"),
    "scores": {"overall": METRICS["overall"], "per_task": METRICS["per_task"]} if METRICS else None,
    "notes": ["Scores are entered by a human reviewer; nothing in this file is auto-scored.",
              "Unscored questions are excluded from averages, never counted as zero.",
              "Partial results are labelled scoring_status='partial'."],
}
atomic_write_text(SUMMARY_JSON, json.dumps(SUMMARY, indent=2, ensure_ascii=False))

if METRICS:
    tbl = []
    for g, b in METRICS["per_task"].items():
        tbl.append({"task": g, "n_total": b["n_total"], "n_scored": b["n_scored"], "n_missing": b["n_missing"], "sum_score": b["sum_score"],
                    "max_possible_scored": b["max_possible_scored"], "mean_score_0_4": b["mean_0_4"], "percent": b["percent"],
                    **{f"count_score_{k}": b["distribution"][k] for k in "01234"}})
    o = METRICS["overall"]
    tbl.append({"task": "OVERALL", "n_total": o["n_total"], "n_scored": o["n_scored"], "n_missing": o["n_missing"], "sum_score": o["sum_score"],
                "max_possible_scored": o["max_possible_scored"], "mean_score_0_4": o["mean_0_4"], "percent": o["percent"],
                **{f"count_score_{k}": o["distribution"][k] for k in "01234"}})
    pd.DataFrame(tbl).to_csv(TASK_SUMMARY_CSV, index=False, encoding="utf-8")
    print("Saved:", SUMMARY_JSON.name, "and", TASK_SUMMARY_CSV.name)

    names = [g for g, b in METRICS["per_task"].items() if b["n_scored"] > 0]
    if names:
        means = [METRICS["per_task"][g]["mean_0_4"] for g in names]
        fig, ax = plt.subplots(figsize=(8, 4.5))
        bars = ax.bar([n.replace("_", "\n") for n in names], means, color="#4C72B0")
        ax.set_ylim(0, 4); ax.set_ylabel("Average human score (0-4 scale)")
        ax.set_title(f"{STUDENT_MODEL_ID} · {MODEL_VARIANT}" + ("  [PARTIAL SCORING]" if METRICS["scoring_status"] == "partial" else ""))
        for b_, g in zip(bars, names):
            ax.text(b_.get_x() + b_.get_width() / 2, b_.get_height() + 0.05, f"{METRICS['per_task'][g]['mean_0_4']:.2f}\n(n={METRICS['per_task'][g]['n_scored']})",
                    ha="center", va="bottom", fontsize=9)
        ax.axhline(4, color="grey", lw=0.5, ls=":")
        fig.tight_layout(); fig.savefig(CHART_PNG, dpi=150); plt.show()
        print("Saved chart:", CHART_PNG.name)
else:
    print(f"Saved {SUMMARY_JSON.name} (generation statistics only). No scores yet, so task_summary.csv and the chart are not created.")

## Cell 15 · Final baseline results and output locations

In [ ]:
print("=" * 78)
print(f" BASELINE RESULTS  ·  {STUDENT_MODEL_ID}  ({MODEL_VARIANT})   run: {RUN_ID}")
print("=" * 78)
print(f" Generation : {GEN_STATS['n_success']}/{GEN_STATS['n_eval_records']} successful, {GEN_STATS['n_failed']} failed, "
      f"{GEN_STATS['n_missing']} missing | truncated {GEN_STATS['n_truncated']} | repetition-flagged {GEN_STATS['n_repetition_flagged']}")
if METRICS and METRICS["scoring_status"] != "not_started":
    o = METRICS["overall"]
    print(f" Scoring    : {METRICS['scoring_status'].upper()}  ({o['n_scored']}/{o['n_total']} scored)")
    for g, b in METRICS["per_task"].items():
        print(f"   {g:26s} mean {b['mean_0_4']}/4  ({b['percent']}%)  n={b['n_scored']}")
    print(f"   {'OVERALL':26s} mean {o['mean_0_4']}/4  ({o['percent']}%)  points {o['sum_score']}/{o['max_possible_scored']} of {o['max_possible_total']} possible")
else:
    print(" Scoring    : NOT STARTED. Score scoring_template.csv by hand, then re-run Cells 13-15.")
print("\n Files in", RESULTS_DIR)
for p in sorted(RESULTS_DIR.glob("*")):
    print(f"   {p.name:34s}{p.stat().st_size:>10,d} bytes")
print("\n Keep these files unchanged: the distilled student must be evaluated with the SAME CSV, prompt format, decoding and rubric.")

## Cell 16 · (Later) Compare with the distilled student

In [ ]:
def compare_runs(baseline_dir, other_dir):
    b = json.loads((Path(baseline_dir) / "summary.json").read_text(encoding="utf-8"))
    o = json.loads((Path(other_dir) / "summary.json").read_text(encoding="utf-8"))
    problems = []
    for key, label in (("eval_csv_sha256", "evaluation CSV"), ("eval_ids_sha1", "evaluation IDs"),
                       ("eval_protocol_sha1", "prompt format / decoding / max tokens / rubric inputs"), ("quantization", "quantization")):
        if b.get(key) != o.get(key):
            problems.append(label)
    if problems:
        raise ValueError("Runs are NOT comparable: they differ in " + ", ".join(problems))
    if b["model_id"] != o["model_id"]:
        print(f"NOTE: different model IDs ({b['model_id']} vs {o['model_id']}); the intended comparison is base vs fine-tuned of the SAME student.")
    for s, name in ((b, "baseline"), (o, "other")):
        if s["scoring_status"] != "complete":
            print(f"WARNING: {name} scoring is {s['scoring_status']}: differences below are provisional.")
    print(f"{'task':26s}{'baseline':>10s}{'other':>10s}{'delta':>9s}")
    for g in list(b["scores"]["per_task"]) + ["OVERALL"]:
        bs = b["scores"]["overall"] if g == "OVERALL" else b["scores"]["per_task"][g]
        os_ = o["scores"]["overall"] if g == "OVERALL" else o["scores"]["per_task"].get(g)
        if bs["mean_0_4"] is None or not os_ or os_["mean_0_4"] is None:
            print(f"{g:26s}{'n/a':>10s}"); continue
        print(f"{g:26s}{bs['mean_0_4']:>10.2f}{os_['mean_0_4']:>10.2f}{os_['mean_0_4'] - bs['mean_0_4']:>+9.2f}")

# Example (after the distilled run exists):
# compare_runs(PROJECT_DIR / "outputs" / "baseline_results", PROJECT_DIR / "outputs" / "distilled_results")